In [7]:
import warnings
warnings.filterwarnings("ignore")
from mlflow import MlflowClient, set_tracking_uri
import mlflow
from typing import Tuple
from tqdm import tqdm
import pandas as pd
from datetime import datetime, timedelta
import mysql.connector
import pyarrow
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import argparse
import os
from dateutil.relativedelta import relativedelta

from functions_daily import get_cutoff_indices, transform_ts_data_into_features_and_target, train_test_split, ts_into_features_Daily, get_data
from ft_tables import load_data
from functions import charge_model
from train_tsif_model_daily import mlflow_daily, mlflow_daily_register





In [8]:
def predict_daily(exchange):
    
    model = None

    # IS THERE ANY MODEL? 
    try:
        model = charge_model(exchange)
    
    
    except Exception as e:
        print(f"{e}")
        return None
    
    # NO MODEL
    if model is None:
        
        print("There is no model")
        
        df_original = get_data(exchange)
        
        # NO DATA
        if df_original == None:
            
            #Load the data:
            load_data(exchange)
            print("Data loaded")
            
            #Train and register the model:
            mlflow_daily_register(exchange)
            
            #Charge the model:
            model = charge_model(exchange)
            
            #Get the data to predict:
            X_test_only_numeric, X_train_only_numeric, y_test, y_train = ts_into_features_Daily(exchange)
            
            #Predict
            predictions = model.predict(pd.DataFrame(X_test_only_numeric))
            return predictions 
        
        # DATA 
        else:
            #Train and register the model:
            mlflow_daily_register(exchange)
            
            #Charge the model:
            model = charge_model(exchange)
            
            #Get the data to predict:
            X_test_only_numeric, X_train_only_numeric, y_test, y_train = ts_into_features_Daily(exchange)
            
            #Predict
            predictions = model.predict(pd.DataFrame(X_test_only_numeric))
            return predictions 
            
    # MODEL       
    else:
        
        print("There is a model registered")
        
        df_original = get_data(exchange)
        
        # NO DATA
        if df_original == None:
            
            # Load the data:
            load_data(exchange)
            print("Data loaded")
            
            # Get data
            X_test_only_numeric, X_train_only_numeric, y_test, y_train = ts_into_features_Daily(exchange)
            
            # Predict
            predictions = model.predict(pd.DataFrame(X_test_only_numeric))
            return predictions 
            
            
        # DATA
        else:
            
            # Get data
            X_test_only_numeric, X_train_only_numeric, y_test, y_train = ts_into_features_Daily(exchange)
            
            # Predict
            predictions = model.predict(pd.DataFrame(X_test_only_numeric))
            return predictions 


In [9]:
exchange = 'BNB-EUR'

predictions = predict_daily(exchange)

predictions

Error loading model: RESOURCE_DOES_NOT_EXIST: Registered Model with name=BNB-EUR_Daily_Model not found
There is no model
MySQL DB Connected
There is no data


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


MySQL DB Connected
New exchange inserted
MySQL DB Connected
Data inserted correctly on the DAILY_DATA table.
MySQL DB Connected
Data inserted correctly on the HOUR_DATA table.
MySQL DB Connected
Data inserted correctly on the FEAR_DATA table.
Data loaded
MySQL DB Connected


100%|██████████| 1/1 [00:01<00:00,  1.57s/it]
2024/07/21 16:50:46 INFO mlflow.tracking.fluent: Experiment with name 'ts_into_features_Daily_BNB-EUR' does not exist. Creating a new experiment.
2024/07/21 16:51:52 WARNING mlflow.models.model: Logging model metadata to the tracking server has failed. The model artifacts have been logged successfully under mlflow-artifacts:/971164910293826610/ea8442abe31442b9bed65fbf80336e0a/artifacts. Set logging level to DEBUG via `logging.getLogger("mlflow").setLevel(logging.DEBUG)` to see the full traceback.
Successfully registered model 'BNB-EUR_Daily_Model'.
2024/07/21 16:51:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BNB-EUR_Daily_Model, version 1
Created version '1' of model 'BNB-EUR_Daily_Model'.


Run: ts_into_features_Daily_LR - BNB-EUR


Model loaded successfully.
MySQL DB Connected


100%|██████████| 1/1 [00:01<00:00,  1.55s/it]


array([550.7994 , 547.13995, 525.0077 , 517.9606 , 539.56836, 550.0979 ,
       551.47546, 543.59656, 537.3208 , 545.5981 , 548.9928 , 542.74817,
       548.97144, 550.7094 , 547.3425 , 527.82855, 533.79803, 523.76013,
       533.2623 , 537.48834, 528.6261 , 551.5362 , 568.24457, 568.4297 ,
       555.86536, 548.3813 , 553.9275 , 553.51807, 555.4076 , 553.975  ,
       553.9116 , 547.4201 , 543.41425, 555.3857 , 555.5366 , 570.8028 ,
       627.7547 , 645.90485, 655.164  , 634.11237, 623.9906 , 625.43604,
       586.241  , 561.0697 , 566.8092 , 564.9051 , 562.99115, 562.12805,
       569.55676, 564.18134, 546.3853 , 551.79193, 551.459  , 552.7083 ,
       548.3207 , 537.4097 , 532.8901 , 534.7333 , 534.4975 , 541.45306,
       534.21716, 533.06647, 540.39197, 540.4912 , 541.8773 , 514.09235,
       473.60376, 457.14563, 477.71558, 457.46954, 468.6518 , 477.9491 ,
       478.70538, 486.1413 , 486.4046 , 485.82104, 498.3236 , 536.9545 ,
       532.8862 ], dtype=float32)